# 2D Pose Evaluation Notebook

This notebook evaluates the performance of various 2D pose estimation models against ground truth data.

**Steps:**
1. Load ground truth annotations from the `labeled_poses` folder.
2. Detect persons using the YOLOv11l model.
3. Run the following pose estimation models:
   - Sapiens-1B
   - RTMPose-l Wholebody
   - Additional models specified in a configuration array.
4. Use the RTMPose Hand model on cropped hand regions from the wholebody pose model.
5. Evaluate predictions against ground truth using COCO metrics (AP and AR).

In [1]:
# Import Required Libraries
import os
import json
import cv2
import numpy as np
from tqdm.notebook import tqdm
from mmpose.apis import init_model, inference_topdown
from mmpose.evaluation.functional import nms
from pycocotools.coco import COCO
from pycocotools.cocoeval import COCOeval

# Import visualization and helper functions from predictors.py
from helpers.predictors import initialize_models, detect_bbox_yolo, estimate_pose_sapiens, detect_body, init_pose_estimator

import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle

print("Libraries and helpers imported successfully.")

Device: cuda
Torch compile enabled: True
Libraries and helpers imported successfully.


In [2]:
# Configuration

# Paths
dataset = 'cha'  # dataset name
data_collection = f'{dataset}/synced_cut'  # dataset collection name
# Path to labeled 2D hand poses (ground truth)
labeled_2d_path = f'labeled_poses/{data_collection}/hand_poses_2d_labeled.npz'  # Update as needed
labeled_frames = list(range(0, 300, 30))
input_dir = f'inputs/{data_collection}/'  # Directory containing images

original_resolution = (3840, 2160)
DEVICE = 'cuda:0'

# Person Detection Model
PERSON_DET_MODEL_CONFIG = 'configs/yolo/yolo11l_config.py'
PERSON_DET_MODEL_CHECKPOINT = 'checkpoints/yolo/yolo11l.pt'

# Pose Models to Evaluate
POSE_MODELS_TO_EVALUATE = {
    'Sapiens-1B': {
        'config': 'configs/wholebody_2d_keypoint/sapiens/sapiens_1b-210e_coco_wholebody-1024x768.py',
        'checkpoint': 'checkpoints/sapiens/sapiens_1b_coco_wholebody_best_coco_wholebody_AP_727.pth'
    },
    'RTMPose-L-WholeBody': {
        'config': 'configs/wholebody_2d_keypoint/rtmpose/coco-wholebody/rtmpose-l_8xb32-270e_coco-wholebody-384x288.py',
        'checkpoint': 'checkpoints/wholebody/rtmpose-l_simcc-coco-wholebody_pt-aic-coco_270e-384x288-eaeb96c8_20230125.pth'
    }
}

# Hand Pose Model
HAND_MODEL_CONFIG = 'configs/hand_2d_keypoint/rtmpose/hand5/rtmpose-m_8xb256-210e_hand5-256x256.py'
HAND_MODEL_CHECKPOINT = 'checkpoints/hands/pose/rtmpose-m_simcc-hand5_pt-aic-coco_210e-256x256-74fb594_20230320.pth'

# Wholebody-Hand Model Combinations
WHOLEBODY_HAND_COMBINATIONS = {}

# Evaluation Parameters
BBOX_SCORE_THR = 0.5
NMS_THR = 0.3

print("Configuration set.")

Configuration set.


In [3]:
# Helper Function to Extract Hand Bounding Boxes
def get_hand_bbox_from_wholebody(wholebody_keypoints, wholebody_scores, img_shape):
    """Extracts hand bounding boxes from whole-body keypoints."""
    l_wrist, r_wrist = 9, 10  # Indices for left and right wrists in COCO-WholeBody
    hand_bboxes = []

    for wrist_idx in [l_wrist, r_wrist]:
        if wholebody_scores[wrist_idx] > 0.3:  # Confidence threshold
            x, y = wholebody_keypoints[wrist_idx]
            size = 64  # Fixed size for hand bounding box
            x1 = max(0, x - size / 2)
            y1 = max(0, y - size / 2)
            x2 = min(img_shape[1], x + size / 2)
            y2 = min(img_shape[0], y + size / 2)
            hand_bboxes.append([x1, y1, x2, y2])

    return np.array(hand_bboxes)

print("Helper function defined.")

Helper function defined.


In [4]:
# Initialize Models
print("Initializing models...")

# Initialize all models using predictors.py
initialize_models(use_tam=False, use_wholebody=True, use_sapiens=True)

print("Models initialized.")

Initializing models...
Loads checkpoint by local backend from path: checkpoints/hands/detection/cascade_rcnn_x101_64x4d_fpn_20e_onehand10k-dac19597_20201030.pth
Loads checkpoint by local backend from path: checkpoints/body/detection/rtmdet_m_8xb32-100e_coco-obj365-person-235e8209.pth
Loads checkpoint by local backend from path: checkpoints/hands/pose/rtmpose-m_simcc-hand5_pt-aic-coco_210e-256x256-74fb594_20230320.pth
Loads checkpoint by local backend from path: checkpoints/body/pose/rtmpose-x_simcc-body7_pt-body7-halpe26_700e-384x288-7fb6e239_20230606.pth
Initializing SapiensPose 1B model...
Loads checkpoint by local backend from path: checkpoints/sapiens/sapiens_1b_coco_wholebody_best_coco_wholebody_AP_727.pth
The model and loaded state dict do not match exactly

missing keys in source state_dict: head.deconv_layers.1.weight, head.deconv_layers.1.bias, head.deconv_layers.1.running_mean, head.deconv_layers.1.running_var, head.deconv_layers.4.weight, head.deconv_layers.4.bias, head.deco

c:\Users\aelvy\miniconda3\envs\orpose\Lib\site-packages\mmengine\utils\manager.py:113: UserWarning: <class 'mmpose.visualization.local_visualizer.PoseLocalVisualizer'> instance named of visualizer has been created, the method `get_instance` should not accept any other arguments
  warnings.warn(


Models initialized with SAM tracker
Pose estimation models initialized
SapiensPose 1B wholebody model ready
YOLO models initialized
Models initialized.


In [5]:
# Load 2D Labeled Hand Poses
data_2d = np.load(labeled_2d_path, allow_pickle=True)
# Assume the file contains a dict: {frame_idx: {cam_name: {obj_id: {'keypoints': ndarray, 'scores': ndarray}}}}
unindexed_labeled_2d_poses = data_2d['poses_2d'].item() if 'poses_2d' in data_2d else data_2d[list(data_2d.keys())[0]].item()
cam_names = list(unindexed_labeled_2d_poses.keys())
labeled_2d_poses = {k: {cam_name: {} for cam_name in cam_names} for k in labeled_frames}
for cam_name in cam_names:
    for frame_idx in labeled_frames:
        for obj_id in unindexed_labeled_2d_poses[cam_name][frame_idx].keys():
            keypoints = unindexed_labeled_2d_poses[cam_name][frame_idx][obj_id]['keypoints']
            scores = unindexed_labeled_2d_poses[cam_name][frame_idx][obj_id]['keypoint_scores']
            # Ensure keypoints and scores are numpy arrays
            labeled_2d_poses[frame_idx][cam_name][obj_id] = {'keypoints':np.array(keypoints), 'scores': np.array(scores)}
print(f"Loaded labeled 2D hand poses for {len(labeled_2d_poses)} frames.")

Loaded labeled 2D hand poses for 10 frames.


In [ ]:
all_model_predictions = {name: [] for name in POSE_MODELS_TO_EVALUATE.keys()}

# Main Processing Loop
for cam_name in cam_names:
    video_name = os.path.join(input_dir, cam_name + '_synced_cut')

    for frame_idx in labeled_frames:
        det_results = detect_body(os.path.join(video_name, f'{frame_idx:05d}.jpg'))
        # Estimate wholebody pose
        bbox = np.array(det_results)
        if len(bbox.shape) == 1:
            bbox = bbox.reshape((1, 4))

        # Read the frame
        img = cv2.imread(os.path.join(video_name, f'{frame_idx:05d}.jpg'))
        if img is None:
            continue
        img_shape = img.shape[:2]

        # Person Detection using predictors.py

        for model_name, pose_model in POSE_MODELS_TO_EVALUATE.items():
            # Pose Estimation
            if model_name == 'Sapiens-1B':
                pose_results = estimate_pose_sapiens(img, bbox)
                
            else:
                pose_estimator_wholebody = init_pose_estimator(
                    pose_model['config'],
                    pose_model['checkpoint'],
                    device='cuda',
                    cfg_options=dict(model=dict(test_cfg=dict(output_heatmaps=False))))
                print(f"{model_name} wholebody model initialized")
                pose_results = inference_topdown(pose_estimator_wholebody, img, bbox)

            # Hand Pose Estimation
            if model_name in WHOLEBODY_HAND_COMBINATIONS:
                for result in pose_results:
                    hand_bboxes = get_hand_bbox_from_wholebody(result.pred_instances.keypoints, result.pred_instances.keypoint_scores, img_shape)
                    if len(hand_bboxes) > 0:
                        hand_pose_results = inference_topdown(hand_model, img, hand_bboxes)
                        # TODO: Merge hand_pose_results with pose_results

            # Format Predictions for Evaluation
            for result in pose_results:
                keypoints = result.pred_instances.keypoints[0]
                scores = result.pred_instances.keypoint_scores[0]
                coco_kpts = [kp for kp, score in zip(keypoints, scores) for kp in [kp[0], kp[1], score]]

                all_model_predictions[model_name].append({
                    'keypoints': coco_kpts,
                    'score': result.pred_instances.bbox_scores[0].item()
                })

        # Visualization
        fig, ax = plt.subplots(1, figsize=(12, 8))
        ax.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
        for bbox in det_results:
            x1, y1, x2, y2 = bbox.squeeze()
            rect = Rectangle((x1, y1), x2 - x1, y2 - y1, linewidth=2, edgecolor='r', facecolor='none')
            ax.add_patch(rect)
        plt.show()

print("Processing complete.")

c:\Users\aelvy\miniconda3\envs\orpose\Lib\site-packages\mmdet\models\layers\se_layer.py:158: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=False):
c:\Users\aelvy\miniconda3\envs\orpose\Lib\site-packages\mmdet\models\backbones\csp_darknet.py:118: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=False):
c:\Users\aelvy\miniconda3\envs\orpose\Lib\site-packages\torch\functional.py:539: UserWarning: torch.meshgrid: in an upcoming release, it will be required to pass the indexing argument. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\pytorch\aten\src\ATen\native\TensorShape.cpp:3638.)
  return _VF.meshgrid(tensors, **kwargs)  # type: ignore[attr-defined]


In [ ]:
# Evaluate Predictions
def calculate_ap_ar(predictions, ground_truths, iou_threshold=0.5):
    """Calculate Average Precision (AP) and Average Recall (AR) for pose estimation."""
    tp, fp, fn = 0, 0, 0

    for gt, pred in zip(ground_truths, predictions):
        matched = set()
        for pred_kpt in pred['keypoints']:
            best_iou = 0
            best_gt_idx = -1
            for gt_idx, gt_kpt in enumerate(gt['keypoints']):
                if gt_idx in matched:
                    continue
                iou = calculate_iou(pred_kpt, gt_kpt)  # Implement IoU calculation for keypoints
                if iou > best_iou:
                    best_iou = iou
                    best_gt_idx = gt_idx

            if best_iou >= iou_threshold:
                tp += 1
                matched.add(best_gt_idx)
            else:
                fp += 1

        fn += len(gt['keypoints']) - len(matched)

    precision = tp / (tp + fp) if (tp + fp) > 0 else 0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0

    return precision, recall

# Visualization of Results
for model_name, predictions in all_model_predictions.items():
    print(f"\n{'='*20} Visualization for {model_name} {'='*20}")
    for pred in predictions:
        img_path = os.path.join(IMAGE_ROOT, pred['image_id'])
        img = cv2.imread(img_path)
        fig, ax = plt.subplots(1, figsize=(12, 8))
        ax.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))

        # Draw keypoints
        for kp in pred['keypoints']:
            x, y, v = kp
            if v > 0:  # Only visualize visible keypoints
                ax.scatter(x, y, c='r', s=10)

        plt.show()

# Custom Evaluation
for model_name, predictions in all_model_predictions.items():
    print(f"\n{'='*20} Evaluation Results for {model_name} {'='*20}")
    precision, recall = calculate_ap_ar(predictions, coco_gt)
    print(f"Precision: {precision:.4f}, Recall: {recall:.4f}")